# Exploratory Data Analysis (EDA) — AI Engineering Interview Prep

Reusable EDA template: missing values, distributions, correlations, outliers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

np.random.seed(42)
sns.set_theme(style="whitegrid")

# Load dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['target_name'] = df['target'].map({0: 'malignant', 1: 'benign'})

# Introduce artificial missings
for col in df.columns[:5]:
    df.loc[np.random.choice(df.index, 10), col] = np.nan

print(f"Shape: {df.shape}")
df.head(3)

## 1. Dataset Overview

In [ ]:
# Comprehensive summary
def eda_overview(df):
    print(f"=== DATASET OVERVIEW ===")
    print(f"Shape:       {df.shape}")
    print(f"Memory:      {df.memory_usage(deep=True).sum()/1e6:.2f} MB")
    print(f"\nDtypes:")
    print(df.dtypes.value_counts())
    print(f"\nMissing values:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    missing_df = pd.concat([missing, missing_pct], axis=1, keys=['count', '%'])
    print(missing_df[missing_df['count'] > 0])
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    print(f"\nTarget distribution:")
    if 'target_name' in df.columns:
        print(df['target_name'].value_counts())

eda_overview(df)

In [ ]:
# Descriptive statistics
desc = df.describe().T
desc['cv'] = desc['std'] / desc['mean']  # coefficient of variation
desc['skew'] = df.select_dtypes('number').skew()
print(desc[['mean', 'std', 'cv', 'min', 'max', 'skew']].round(3))

## 2. Missing Value Analysis

In [ ]:
missing_cols = df.columns[df.isnull().any()].tolist()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Missing by column
miss_pct = df[missing_cols].isnull().mean() * 100
miss_pct.sort_values().plot(kind='barh', ax=axes[0], color='salmon')
axes[0].set_title("Missing Values (%)")
axes[0].axvline(5, color='red', linestyle='--', alpha=0.5)

# Missing pattern heatmap (sample)
sns.heatmap(df[missing_cols].isnull().head(100), ax=axes[1],
            cmap='YlOrRd', cbar=False, yticklabels=False)
axes[1].set_title("Missingness Pattern (first 100 rows)")

plt.tight_layout()
plt.show()

## 3. Distribution Analysis

In [ ]:
num_cols = df.select_dtypes('number').columns[:6].tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for ax, col in zip(axes, num_cols):
    for label, color in [('malignant', 'salmon'), ('benign', 'steelblue')]:
        subset = df[df['target_name'] == label][col].dropna()
        ax.hist(subset, bins=25, alpha=0.5, color=color, label=label, density=True)
    ax.set_title(col[:30])
    ax.legend(fontsize=7)

plt.suptitle("Feature Distributions by Target Class", y=1.01)
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

In [ ]:
corr = df.select_dtypes('number').corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5)
plt.title("Feature Correlation Matrix (lower triangle)")
plt.tight_layout()
plt.show()

# Highly correlated pairs
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        val = corr.iloc[i, j]
        if abs(val) > 0.9:
            high_corr.append((corr.columns[i], corr.columns[j], round(val, 3)))

print(f"\nHighly correlated pairs (|r|>0.9): {len(high_corr)}")
for pair in high_corr[:5]:
    print(f"  {pair[0]} <-> {pair[1]}: r={pair[2]}")

## 5. Outlier Detection

In [ ]:
# IQR-based outlier counts per feature
def count_outliers_iqr(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((series < q1 - 1.5*iqr) | (series > q3 + 1.5*iqr)).sum()

num_features = df.select_dtypes('number').drop(columns=['target'], errors='ignore')
outlier_counts = num_features.apply(count_outliers_iqr)
print("Outlier counts (IQR method):")
print(outlier_counts.sort_values(ascending=False).head(10))

# Box plots for top outlier features
top_cols = outlier_counts.sort_values(ascending=False).head(4).index.tolist()
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col in zip(axes, top_cols):
    df.boxplot(column=col, by='target_name', ax=ax)
    ax.set_title(col[:20])
    ax.set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Interview Tips

- **EDA order**: shape → dtypes → missing → distributions → correlations → outliers → target relationship.
- **High correlation (|r|>0.9)**: signals multicollinearity; consider dropping one or using PCA.
- **Class imbalance**: check target distribution early — affects model choice and evaluation metrics.
- **Data leakage**: features that reveal the target (e.g., derived from it) — common interview trap.
- **skew > 2**: consider log transform for right-skewed numeric features before modelling.
- **Missing completely at random (MCAR) vs MAR vs MNAR**: drives imputation strategy choice.